# Day 012 | Modules & Packages

## 100 Days of ML for Geotechnical Engineering
**Phase:** Python Fundamentals | **Author:** Ripon Chandra Malo | **Date:** February 27, 2026

---

## What We Will Learn Today

| Concept | What It Means | Geotechnical Example |
|---------|--------------|---------------------|
| `import` module | Load external code | `import math` for slope-angle trig |
| `from...import` | Grab specific tools | `from statistics import mean` for N-values |
| `import...as` | Give a short alias | `import numpy as np` |
| Standard Library | Python's built-in toolbox | `math`, `statistics`, `csv`, `json`, `os` |
| Third-party packages | Install with `pip` | `numpy`, `pandas`, `scipy` |
| Custom modules | Your own `.py` files | `geotech_utils.py` with bearing capacity functions |
| `__name__ == "__main__"` | Run vs import guard | Test module standalone, import without side effects |

---

## Our Geotechnical Problem

A site investigation produces **dozens of calculations** that get reused across projects:

- SPT correction factors (energy, overburden, rod length)
- Bearing capacity from Terzaghi or Meyerhof
- Shear-wave velocity from correlations
- Soil classification from grain-size data

Without modules, you **copy-paste** these functions into every new script. One bug fix means editing 15 files.

With modules, you write each function **once** in a shared `.py` file, and every project imports it.

### The Tool Shed Analogy

```
Without modules:                  With modules:
  Every project carries            Shared tool shed:
  its own copy of every tool         geotech_utils.py
                                   
  Project A: bearing_cap()         Project A: import geotech_utils
  Project B: bearing_cap()  -->    Project B: import geotech_utils
  Project C: bearing_cap()         Project C: import geotech_utils
  (3 copies, 3 places to fix)      (1 source of truth)
```

---

## Part 1: What Are Modules? Three Ways to Import

A **module** is simply a `.py` file containing functions, classes, and variables. Python ships with hundreds of built-in modules called the **Standard Library**.

| Import Style | Syntax | When to Use |
|-------------|--------|-------------|
| Full module | `import math` | When you use many things from it |
| Specific item | `from math import sqrt` | When you need just one or two things |
| With alias | `import numpy as np` | Long module names you type often |

In [ ]:
# ─────────────────────────────────────────────────────
# THREE WAYS TO IMPORT — slope stability example
# ─────────────────────────────────────────────────────

# Style 1: import the whole module
import math

slope_angle_deg = 30  # slope face angle
slope_angle_rad = math.radians(slope_angle_deg)

driving_component = math.sin(slope_angle_rad)
resisting_component = math.cos(slope_angle_rad)

print("Style 1: import math")
print(f"  Slope angle: {slope_angle_deg} degrees")
print(f"  sin({slope_angle_deg}) = {driving_component:.4f}  (drives sliding)")
print(f"  cos({slope_angle_deg}) = {resisting_component:.4f}  (resists sliding)")
print()

# ─────────────────────────────────────────────────────
# Style 2: from module import specific functions
from math import sin, cos, radians, sqrt, tan

phi = 35              # friction angle (degrees)
c = 10                # cohesion (kPa)
gamma = 18.5          # unit weight (kN/m3)
depth = 2.0           # depth of failure plane (m)
alpha = 25            # slope angle (degrees)

# Infinite slope stability (no water)
normal_stress = gamma * depth * cos(radians(alpha))
shear_strength = c + normal_stress * tan(radians(phi))
driving_stress = gamma * depth * sin(radians(alpha))
FoS = shear_strength / driving_stress

print("Style 2: from math import sin, cos, radians")
print(f"  Normal stress:  {normal_stress:.2f} kPa")
print(f"  Shear strength: {shear_strength:.2f} kPa")
print(f"  Driving stress: {driving_stress:.2f} kPa")
print(f"  Factor of Safety: {FoS:.2f}")
print()

# ─────────────────────────────────────────────────────
# Style 3: import with alias
import math as m

H = 8.0  # excavation depth (m)
phi_rad = m.radians(30)
Ka = m.tan(m.pi / 4 - phi_rad / 2) ** 2  # Rankine active pressure coeff

print("Style 3: import math as m")
print(f"  Ka (Rankine, phi=30): {Ka:.4f}")
print(f"  Active pressure at {H}m: {0.5 * gamma * H**2 * Ka:.1f} kN/m")

---

## Part 2: The `math` Module — Trigonometry for Foundations

The `math` module is an engineer's best friend. Key functions:

| Function | What It Does | Geotech Use |
|----------|-------------|-------------|
| `sin()`, `cos()`, `tan()` | Trig (radians) | Slope forces, earth pressure |
| `radians()`, `degrees()` | Angle conversion | Field angles are in degrees |
| `sqrt()` | Square root | Bearing capacity factors |
| `exp()` | e^x | Consolidation, permeability |
| `log()`, `log10()` | Logarithms | Cc from e-log(p) curve |
| `pi` | 3.14159... | Pile circumference, circular footing |

In [ ]:
# ─────────────────────────────────────────────────────
# math MODULE — Bearing capacity factors (Terzaghi)
# ─────────────────────────────────────────────────────
import math

def terzaghi_factors(phi_deg):
    """
    Compute Terzaghi bearing capacity factors Nc, Nq, Ngamma.
    Uses math module for trig and exponential functions.
    """
    phi = math.radians(phi_deg)
    
    # Nq = exp(2*pi*(0.75 - phi/360)*tan(phi)) / (2*cos^2(45+phi/2))
    Nq = math.exp(2 * math.pi * (0.75 - phi_deg / 360) * math.tan(phi))
    Nq /= (2 * math.cos(math.radians(45 + phi_deg / 2)) ** 2)
    
    # Nc = (Nq - 1) * cot(phi)
    if phi_deg > 0:
        Nc = (Nq - 1) / math.tan(phi)
    else:
        Nc = 5.7  # special case for phi=0
    
    # Ngamma (Kumbhojkar 1993 approximation)
    Ngamma = 2 * (Nq + 1) * math.tan(phi)
    
    return Nc, Nq, Ngamma

# Calculate for common friction angles
print("Terzaghi Bearing Capacity Factors")
print(f"{'phi':>5}  {'Nc':>8}  {'Nq':>8}  {'Ngamma':>8}")
print("-" * 35)

for phi in [0, 10, 20, 25, 30, 35, 40]:
    Nc, Nq, Ng = terzaghi_factors(phi)
    print(f"{phi:5d}  {Nc:8.2f}  {Nq:8.2f}  {Ng:8.2f}")

print()
print("All calculations used: math.exp, math.tan, math.cos, math.radians, math.pi")

---

## Part 3: The `statistics` Module — Quick Stats on Field Data

No need for numpy just to get a mean or standard deviation. The `statistics` module handles basic stats cleanly.

In [ ]:
# ─────────────────────────────────────────────────────
# statistics MODULE — Analyzing SPT blow counts
# ─────────────────────────────────────────────────────
from statistics import mean, median, stdev, variance

# SPT N-values from three boreholes at a site
bh01_N = [5, 8, 12, 18, 25, 30, 35]
bh02_N = [3, 6, 10, 15, 22, 28, 40]
bh03_N = [7, 9, 14, 20, 27, 33, 38]

all_N = bh01_N + bh02_N + bh03_N

print("SPT N-value Statistics Across 3 Boreholes")
print("=" * 45)

for name, data in [("BH-01", bh01_N), ("BH-02", bh02_N),
                    ("BH-03", bh03_N), ("ALL",   all_N)]:
    print(f"\n  {name}:")
    print(f"    Mean:   {mean(data):6.1f}")
    print(f"    Median: {median(data):6.1f}")
    print(f"    Stdev:  {stdev(data):6.1f}")
    print(f"    CoV:    {stdev(data)/mean(data)*100:5.1f}%")

print("\n" + "-" * 45)
print("CoV > 30% suggests high spatial variability.")
print("This affects characteristic value selection (Eurocode 7).")

---

## Part 4: `csv` and `json` — Reading & Storing Engineering Data

Two modules you will use constantly:

| Module | Format | Geotech Use |
|--------|--------|-------------|
| `csv` | Comma-separated values | Borehole logs, lab results, monitoring data |
| `json` | Structured key-value | Test configurations, project metadata, API data |

In [ ]:
# ─────────────────────────────────────────────────────
# csv MODULE — Reading borehole data
# ─────────────────────────────────────────────────────
import csv
import os

os.makedirs("project_data", exist_ok=True)

# Write a borehole CSV
borehole_data = [
    ["depth_m", "N60", "soil_type", "moisture_pct"],
    [1.5, 5, "Fill", 18.2],
    [3.0, 8, "Silty Clay", 32.5],
    [4.5, 15, "Clayey Sand", 22.1],
    [6.0, 22, "Sand", 16.8],
    [7.5, 30, "Sand", 14.3],
    [9.0, 45, "Gravelly Sand", 11.7],
]

with open("project_data/bh01.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerows(borehole_data)

# Read it back with DictReader
print("Reading borehole CSV with csv.DictReader:")
print(f"{'Depth':>6} {'N60':>4} {'Soil':15s} {'w%':>6}")
print("-" * 35)

with open("project_data/bh01.csv", "r") as f:
    reader = csv.DictReader(f)
    for row in reader:
        print(f"{float(row['depth_m']):5.1f}m {row['N60']:>4} "
              f"{row['soil_type']:15s} {row['moisture_pct']:>5}%")

In [ ]:
# ─────────────────────────────────────────────────────
# json MODULE — Storing project configurations
# ─────────────────────────────────────────────────────
import json

# Project configuration as a dictionary
project_config = {
    "project_name": "Highway Bridge Foundation",
    "project_id": "GE-2026-042",
    "client": "Dept of Transportation",
    "location": {"lat": 23.8103, "lon": 90.4125},
    "boreholes": ["BH-01", "BH-02", "BH-03"],
    "water_table_m": 3.2,
    "design_params": {
        "bearing_pressure_kPa": 150,
        "settlement_limit_mm": 25,
        "FoS_required": 3.0
    },
    "standards": ["ASTM D1586", "ASTM D2487", "IS 6403"]
}

# Save to JSON
with open("project_data/project_config.json", "w") as f:
    json.dump(project_config, f, indent=2)
print("Saved project_config.json")

# Load from JSON
with open("project_data/project_config.json", "r") as f:
    loaded = json.load(f)

print(f"\nProject: {loaded['project_name']}")
print(f"Boreholes: {', '.join(loaded['boreholes'])}")
print(f"Water table: {loaded['water_table_m']} m")
print(f"Required FoS: {loaded['design_params']['FoS_required']}")
print(f"Standards: {', '.join(loaded['standards'])}")
print()
print("JSON is perfect for project metadata — readable, shareable, version-controllable.")

---

## Part 5: `os` and `pathlib` — Organizing Project Files

Real projects have folder structures. These modules help you navigate and create them programmatically.

In [ ]:
# ─────────────────────────────────────────────────────
# os and pathlib — Project directory management
# ─────────────────────────────────────────────────────
import os
from pathlib import Path

# Create a typical geotechnical project structure
project_root = Path("demo_project")
folders = [
    "01_field_data/boreholes",
    "01_field_data/lab_results",
    "02_analysis/bearing_capacity",
    "02_analysis/settlement",
    "03_reports",
    "04_drawings",
]

for folder in folders:
    (project_root / folder).mkdir(parents=True, exist_ok=True)

print("Project structure created:")
for folder in sorted(project_root.rglob("*")):
    if folder.is_dir():
        depth = len(folder.relative_to(project_root).parts)
        indent = "  " * depth
        print(f"{indent}{folder.name}/")

print()

# ─── Useful os / pathlib operations ───
print("Useful path operations:")
bh_path = project_root / "01_field_data" / "boreholes" / "BH-01.csv"
print(f"  Build path: {bh_path}")
print(f"  File name:  {bh_path.name}")
print(f"  Extension:  {bh_path.suffix}")
print(f"  Parent dir: {bh_path.parent}")
print(f"  Exists?     {bh_path.exists()}")
print(f"  CWD:        {Path.cwd().name}")
print()
print("pathlib makes file paths clean and cross-platform (Windows/Mac/Linux).")

---

## Part 6: `datetime` — Timestamps for Monitoring & Logs

Geotechnical monitoring (settlement plates, piezometers, inclinometers) produces time-stamped data. The `datetime` module handles dates and time intervals.

In [ ]:
# ─────────────────────────────────────────────────────
# datetime MODULE — Settlement monitoring timeline
# ─────────────────────────────────────────────────────
from datetime import datetime, timedelta
import math

# Project timeline
construction_start = datetime(2026, 3, 15)
fill_placement     = datetime(2026, 4, 1)
monitoring_start   = datetime(2026, 4, 5)

# Simulated settlement readings (date, settlement_mm)
readings = []
S_final = 45.0  # mm (expected total settlement)
tau = 60.0      # days (time constant for consolidation)

for week in range(8):
    date = monitoring_start + timedelta(weeks=week)
    days_since_load = (date - fill_placement).days
    # Simplified consolidation curve: S = S_final * (1 - exp(-t/tau))
    settlement = S_final * (1 - math.exp(-days_since_load / tau))
    readings.append((date, days_since_load, round(settlement, 1)))

print("Settlement Monitoring Log")
print(f"Construction start: {construction_start.strftime('%Y-%m-%d')}")
print(f"Fill placed:        {fill_placement.strftime('%Y-%m-%d')}")
print()
print(f"{'Date':12s} {'Days':>5} {'Settlement':>10}")
print("-" * 32)
for date, days, s in readings:
    bar = '#' * int(s / 2)
    print(f"{date.strftime('%Y-%m-%d'):12s} {days:5d} {s:8.1f} mm  {bar}")

t90 = fill_placement + timedelta(days=int(tau * 2.3))
print(f"\n90% consolidation expected by: {t90.strftime('%Y-%m-%d')}")

---

## Part 7: Third-Party Packages — `pip install`

The Standard Library is powerful, but geotechnical ML needs more. Third-party packages are installed with `pip`.

```bash
pip install numpy        # Fast arrays and math
pip install pandas       # DataFrames for tabular data
pip install matplotlib   # Plotting (borehole logs, profiles)
pip install scipy        # Optimization, interpolation, statistics
```

| Package | What It Does | Geotech Use |
|---------|-------------|-------------|
| `numpy` | Fast numerical arrays | SPT profiles, stress distributions |
| `pandas` | DataFrames (tables) | Borehole databases, lab results |
| `matplotlib` | Plotting | Borehole logs, settlement curves |
| `scipy` | Scientific computing | Curve fitting, interpolation |
| `scikit-learn` | Machine learning | Soil classification, prediction |

In [ ]:
# ─────────────────────────────────────────────────────
# PREVIEW: numpy and pandas (we will use these daily soon)
# ─────────────────────────────────────────────────────

# numpy — fast array operations
import numpy as np

depths = np.array([1.5, 3.0, 4.5, 6.0, 7.5, 9.0])
N_values = np.array([5, 8, 15, 22, 30, 45])

# Vectorized calculation — ALL values at once, no loop
Vs = 97 * N_values ** 0.314  # shear wave velocity

print("numpy: Vectorized SPT correlation (no loops!)")
print(f"  N-values: {N_values}")
print(f"  Vs (m/s): {np.round(Vs, 1)}")
print(f"  Mean Vs:  {np.mean(Vs):.1f} m/s")
print()

# pandas — tabular data with labels
import pandas as pd

df = pd.DataFrame({
    "depth_m": depths,
    "N60": N_values,
    "Vs_mps": np.round(Vs, 1),
    "soil": ["Fill", "Silty Clay", "Clayey Sand", "Sand", "Sand", "Gravelly Sand"]
})

print("pandas: Borehole data as a DataFrame")
print(df.to_string(index=False))
print(f"\nWith pandas you can filter, group, merge, and export — all in one line.")

---

## Part 8: Creating Your Own Module

In a real project, you would create a file called `geotech_utils.py` and import it:

```python
# geotech_utils.py
def spt_correction(N, energy_ratio=0.6): ...
def bearing_capacity(c, phi, gamma, B, D): ...
def vs_from_spt(N60): ...
```

Then in your analysis script:
```python
from geotech_utils import bearing_capacity, spt_correction
```

Since we are in a notebook, we will **define the module functions in one cell** and use them in later cells. This is exactly how a `.py` module works — define once, use everywhere.

In [ ]:
# ================================================================
# geotech_utils — Our custom geotechnical module
# ================================================================
# In a real project, this entire cell would be saved as
# geotech_utils.py in your project folder.
# ================================================================

import math

# ─────────────────────────────────────────────────────
# FUNCTION 1: SPT Energy Correction (N60)
# ─────────────────────────────────────────────────────
def spt_correction(N_field, energy_ratio=0.6, rod_length_m=10,
                   borehole_dia_mm=150, sampler="standard"):
    """
    Correct raw SPT N-value to N60.
    
    Parameters:
        N_field: raw blow count from field
        energy_ratio: hammer efficiency (default 0.6 for safety hammer)
        rod_length_m: total rod length
        borehole_dia_mm: borehole diameter
        sampler: 'standard' or 'liner'
    
    Returns:
        N60 corrected blow count
    """
    # Energy correction
    Ce = energy_ratio / 0.60
    
    # Rod length correction
    if rod_length_m < 3:     Cr = 0.75
    elif rod_length_m < 4:   Cr = 0.80
    elif rod_length_m < 6:   Cr = 0.85
    elif rod_length_m < 10:  Cr = 0.95
    else:                    Cr = 1.00
    
    # Borehole diameter correction
    if borehole_dia_mm <= 115:   Cb = 1.00
    elif borehole_dia_mm <= 150: Cb = 1.05
    else:                        Cb = 1.15
    
    # Sampler correction
    Cs = 1.0 if sampler == "standard" else 1.2
    
    N60 = round(N_field * Ce * Cr * Cb * Cs)
    return N60


# ─────────────────────────────────────────────────────
# FUNCTION 2: Shear Wave Velocity from SPT
# ─────────────────────────────────────────────────────
def vs_from_spt(N60, soil_type="all"):
    """
    Estimate shear wave velocity from SPT N60.
    
    Correlations:
        All soils:  Vs = 97 * N60^0.314  (Imai & Tonouchi, 1982)
        Sand:       Vs = 80.6 * N60^0.331 (Ohta & Goto, 1978)
        Clay:       Vs = 114.4 * N60^0.271 (Imai, 1977)
    """
    if N60 <= 0:
        return 0.0
    
    correlations = {
        "all":  (97.0, 0.314),
        "sand": (80.6, 0.331),
        "clay": (114.4, 0.271),
    }
    
    a, b = correlations.get(soil_type.lower(), (97.0, 0.314))
    return round(a * N60 ** b, 1)


# ─────────────────────────────────────────────────────
# FUNCTION 3: Bearing Capacity (Terzaghi)
# ─────────────────────────────────────────────────────
def bearing_capacity(c_kPa, phi_deg, gamma_kNm3, B_m, D_m,
                     footing="strip"):
    """
    Ultimate bearing capacity using Terzaghi's equation.
    
    Returns:
        (q_ult in kPa, q_allow in kPa with FoS=3)
    """
    phi = math.radians(phi_deg)
    
    # Bearing capacity factors
    Nq = math.exp(2 * math.pi * (0.75 - phi_deg / 360) * math.tan(phi))
    Nq /= (2 * math.cos(math.radians(45 + phi_deg / 2)) ** 2)
    
    Nc = (Nq - 1) / math.tan(phi) if phi_deg > 0 else 5.7
    Ngamma = 2 * (Nq + 1) * math.tan(phi)
    
    # Shape factors
    if footing == "strip":
        sc, sq, sg = 1.0, 1.0, 1.0
    elif footing == "square":
        sc, sq, sg = 1.3, 1.0, 0.8
    elif footing == "circular":
        sc, sq, sg = 1.3, 1.0, 0.6
    else:
        sc, sq, sg = 1.0, 1.0, 1.0
    
    q = gamma_kNm3 * D_m  # overburden pressure
    
    q_ult = (sc * c_kPa * Nc +
             sq * q * Nq +
             sg * 0.5 * gamma_kNm3 * B_m * Ngamma)
    
    q_allow = q_ult / 3.0
    return round(q_ult, 1), round(q_allow, 1)


# ─────────────────────────────────────────────────────
# FUNCTION 4: Soil Classification (simplified USCS)
# ─────────────────────────────────────────────────────
def soil_classify(fines_pct, sand_pct, gravel_pct, LL=None, PI=None):
    """
    Simplified USCS classification from grain size and Atterberg limits.
    
    Returns:
        (symbol, description)
    """
    if fines_pct >= 50:
        # Fine-grained
        if LL is not None and PI is not None:
            if LL < 50:
                return ("CL", "Lean Clay") if PI > 7 else ("ML", "Silt")
            else:
                return ("CH", "Fat Clay") if PI > 7 else ("MH", "Elastic Silt")
        return ("Fine", "Fine-grained (need Atterberg limits)")
    else:
        # Coarse-grained
        if gravel_pct > sand_pct:
            if fines_pct < 5:
                return ("GP", "Poorly Graded Gravel")
            elif fines_pct < 12:
                return ("GM", "Silty Gravel")
            else:
                return ("GC", "Clayey Gravel")
        else:
            if fines_pct < 5:
                return ("SP", "Poorly Graded Sand")
            elif fines_pct < 12:
                return ("SM", "Silty Sand")
            else:
                return ("SC", "Clayey Sand")


print("geotech_utils module defined with 4 functions:")
print("  1. spt_correction()   -- N-field to N60")
print("  2. vs_from_spt()      -- Shear wave velocity from N60")
print("  3. bearing_capacity() -- Terzaghi bearing capacity")
print("  4. soil_classify()    -- Simplified USCS classification")
print()
print("In a real project, save this cell as 'geotech_utils.py'")
print("Then use: from geotech_utils import bearing_capacity")

---

## Part 9: Using Our Module — Testing Each Function

Now we use the functions defined above, exactly as if we had written `from geotech_utils import ...`

In [ ]:
# ─────────────────────────────────────────────────────
# TEST 1: SPT Correction — Different hammer types
# ─────────────────────────────────────────────────────

N_field = 20

print("SPT Correction for N_field = 20")
print(f"{'Scenario':30s} {'N60':>5}")
print("-" * 38)

scenarios = [
    ("Safety hammer, deep",      {"energy_ratio": 0.60, "rod_length_m": 15}),
    ("Auto trip, deep",          {"energy_ratio": 0.73, "rod_length_m": 15}),
    ("Donut hammer, shallow",    {"energy_ratio": 0.45, "rod_length_m": 4}),
    ("Safety, large borehole",   {"energy_ratio": 0.60, "borehole_dia_mm": 200}),
]

for name, params in scenarios:
    n60 = spt_correction(N_field, **params)
    print(f"  {name:30s} {n60:5d}")

print("\nSame raw N, different corrected values — corrections matter!")

In [ ]:
# ─────────────────────────────────────────────────────
# TEST 2: Vs Estimation — Compare correlations
# ─────────────────────────────────────────────────────

test_N = [5, 10, 15, 20, 30, 50]

print("Vs Estimation from SPT (comparing correlations)")
print(f"{'N60':>5} {'All soils':>10} {'Sand':>10} {'Clay':>10}")
print("-" * 38)

for n in test_N:
    vs_all  = vs_from_spt(n, "all")
    vs_sand = vs_from_spt(n, "sand")
    vs_clay = vs_from_spt(n, "clay")
    print(f"{n:5d} {vs_all:8.1f} m/s {vs_sand:8.1f} m/s {vs_clay:8.1f} m/s")

In [ ]:
# ─────────────────────────────────────────────────────
# TEST 3: Bearing Capacity — Different footings
# ─────────────────────────────────────────────────────

# Soil: medium dense sand
c = 0        # cohesionless
phi = 30     # degrees
gamma = 18   # kN/m3
B = 2.0      # footing width (m)
D = 1.5      # depth of footing (m)

print(f"Bearing Capacity: c={c} kPa, phi={phi}, gamma={gamma} kN/m3")
print(f"Footing: B={B}m, D={D}m")
print()
print(f"{'Type':10s} {'q_ult (kPa)':>12} {'q_allow (kPa)':>14}")
print("-" * 40)

for shape in ["strip", "square", "circular"]:
    q_ult, q_allow = bearing_capacity(c, phi, gamma, B, D, shape)
    print(f"{shape:10s} {q_ult:12.1f} {q_allow:14.1f}")

print("\nSquare and circular footings have higher capacity (shape factors).")

In [ ]:
# ─────────────────────────────────────────────────────
# TEST 4: Soil Classification
# ─────────────────────────────────────────────────────

samples = [
    {"name": "Sample A", "fines": 3,  "sand": 72, "gravel": 25, "LL": None, "PI": None},
    {"name": "Sample B", "fines": 8,  "sand": 65, "gravel": 27, "LL": None, "PI": None},
    {"name": "Sample C", "fines": 35, "sand": 50, "gravel": 15, "LL": None, "PI": None},
    {"name": "Sample D", "fines": 65, "sand": 25, "gravel": 10, "LL": 38,  "PI": 15},
    {"name": "Sample E", "fines": 72, "sand": 18, "gravel": 10, "LL": 55,  "PI": 30},
    {"name": "Sample F", "fines": 60, "sand": 30, "gravel": 10, "LL": 42,  "PI": 4},
]

print("Soil Classification Results (Simplified USCS)")
print(f"{'Sample':10s} {'Fines%':>6} {'Sand%':>6} {'Grav%':>6} {'Symbol':>7} Description")
print("-" * 62)

for s in samples:
    symbol, desc = soil_classify(s["fines"], s["sand"], s["gravel"],
                                 s["LL"], s["PI"])
    print(f"{s['name']:10s} {s['fines']:6d} {s['sand']:6d} {s['gravel']:6d} "
          f"{symbol:>7s} {desc}")

---

## Part 10: The `__name__ == "__main__"` Pattern

When you create a `.py` module, you often want to:
- **Import it** in other scripts (no side effects)
- **Run it directly** to test it (show output)

The `if __name__ == "__main__":` guard makes this possible:

```python
# geotech_utils.py

def vs_from_spt(N60):
    return 97 * N60 ** 0.314

def spt_correction(N_field, energy_ratio=0.6):
    return round(N_field * energy_ratio / 0.60)

# This block ONLY runs when you execute the file directly:
#   python geotech_utils.py    --> runs the tests below
#   import geotech_utils       --> skips the tests

if __name__ == "__main__":
    print("Testing geotech_utils...")
    print(f"  Vs(N60=20) = {vs_from_spt(20)} m/s")
    print(f"  N60 correction: {spt_correction(25, 0.72)}")
    print("  All tests passed!")
```

| Scenario | `__name__` value | Guard block runs? |
|----------|-----------------|-------------------|
| `python geotech_utils.py` | `"__main__"` | YES |
| `import geotech_utils` | `"geotech_utils"` | NO |

In [ ]:
# ─────────────────────────────────────────────────────
# __name__ == "__main__" — How it works
# ─────────────────────────────────────────────────────

# In a notebook, __name__ is always "__main__"
print(f"Current __name__: {__name__}")
print()

# Demonstrate the guard
if __name__ == "__main__":
    print("This code runs because we are executing directly (notebook = __main__).")
    print()
    print("Quick self-test of our geotech_utils functions:")
    print(f"  Vs(N60=20, all)  = {vs_from_spt(20, 'all')} m/s")
    print(f"  Vs(N60=20, sand) = {vs_from_spt(20, 'sand')} m/s")
    print(f"  N60 correction (N=25, Er=0.72) = {spt_correction(25, energy_ratio=0.72)}")
    print(f"  soil_classify(fines=8, sand=65, gravel=27) = {soil_classify(8, 65, 27)}")
    print()
    print("If this were imported as a module, these lines would be skipped.")
    print("That is how you keep imports clean — no surprise print statements.")

---

## Part 11: Complete Application — Multi-Layer Borehole Analysis

Now we put our `geotech_utils` module to work on a real problem:

**Given:** A 12m deep borehole with 7 layers, raw SPT data, and grain-size information.

**Task:** Correct all N-values, estimate Vs for each layer, classify soils, and compute bearing capacity for a proposed 2m x 2m square footing at 2m depth.

```
Borehole BH-04 | Project: Highway Bridge | Location: Dhaka
-------------------------------------------------------
   0m +-------+
      | Fill  |  N=5
 1.5m +-------+
      | Silty |  N=8
 3.0m + Clay  +  <-- Water table at 3.2m
      |Clayey |  N=14
 4.5m + Sand  +
      | Sand  |  N=20
 6.0m +-------+
      | Sand  |  N=28
 7.5m +-------+
      | Grav. |  N=35
 9.0m + Sand  +
      | Dense |  N=50
10.5m +-------+
```

In [ ]:
# ================================================================
# COMPLETE APPLICATION: Multi-Layer Borehole Analysis
# Using our geotech_utils functions
# ================================================================

# ─── Raw field data ───
borehole = {
    "id": "BH-04",
    "project": "Highway Bridge Foundation",
    "location": "Dhaka, Bangladesh",
    "water_table_m": 3.2,
    "hammer": "safety",
    "energy_ratio": 0.60,
}

layers = [
    {"depth": 1.5,  "N_raw": 5,  "soil": "Fill",
     "fines": 20, "sand": 55, "gravel": 25, "LL": None, "PI": None},
    {"depth": 3.0,  "N_raw": 8,  "soil": "Silty Clay",
     "fines": 68, "sand": 22, "gravel": 10, "LL": 35,   "PI": 12},
    {"depth": 4.5,  "N_raw": 14, "soil": "Clayey Sand",
     "fines": 18, "sand": 62, "gravel": 20, "LL": None, "PI": None},
    {"depth": 6.0,  "N_raw": 20, "soil": "Sand",
     "fines": 4,  "sand": 78, "gravel": 18, "LL": None, "PI": None},
    {"depth": 7.5,  "N_raw": 28, "soil": "Sand",
     "fines": 3,  "sand": 82, "gravel": 15, "LL": None, "PI": None},
    {"depth": 9.0,  "N_raw": 35, "soil": "Gravelly Sand",
     "fines": 5,  "sand": 45, "gravel": 50, "LL": None, "PI": None},
    {"depth": 10.5, "N_raw": 50, "soil": "Dense Sand",
     "fines": 2,  "sand": 70, "gravel": 28, "LL": None, "PI": None},
]

print(f"Borehole: {borehole['id']}")
print(f"Project:  {borehole['project']}")
print(f"Location: {borehole['location']}")
print(f"Water table: {borehole['water_table_m']} m")
print(f"Layers: {len(layers)}")

In [ ]:
# ================================================================
# STEP 1: Correct N-values, estimate Vs, classify soils
# ================================================================

print("STEP 1: SPT Correction, Vs Estimation, Soil Classification")
print("=" * 70)
print(f"{'Depth':>6} {'N_raw':>6} {'N60':>5} {'Vs':>7} {'Corr.':>6} "
      f"{'Soil':15s} {'USCS':>5}")
print("-" * 70)

for layer in layers:
    d = layer["depth"]
    
    # Correct N-value using our module function
    rod_len = d + 1.0  # approximate rod length = depth + above-ground
    layer["N60"] = spt_correction(
        layer["N_raw"],
        energy_ratio=borehole["energy_ratio"],
        rod_length_m=rod_len
    )
    
    # Estimate Vs using soil-specific correlation
    soil_lower = layer["soil"].lower()
    if "clay" in soil_lower:
        vs_type = "clay"
    elif "sand" in soil_lower or "gravel" in soil_lower:
        vs_type = "sand"
    else:
        vs_type = "all"
    
    layer["Vs"] = vs_from_spt(layer["N60"], vs_type)
    layer["vs_type"] = vs_type
    
    # Classify soil using grain-size data
    symbol, desc = soil_classify(
        layer["fines"], layer["sand"], layer["gravel"],
        layer["LL"], layer["PI"]
    )
    layer["uscs"] = symbol
    
    print(f"{d:5.1f}m {layer['N_raw']:6d} {layer['N60']:5d} "
          f"{layer['Vs']:5.1f}  {vs_type:>6s}  {layer['soil']:15s} {symbol:>5s}")

In [ ]:
# ================================================================
# STEP 2: Bearing Capacity for Proposed Footing
# ================================================================

# Design parameters
B_footing = 2.0   # width (m)
D_footing = 2.0   # depth (m)

# Soil at footing depth (~2m): between Fill and Silty Clay layer
# Use conservative parameters for silty clay
c_design = 15        # kPa (undrained cohesion)
phi_design = 20      # degrees
gamma_design = 17.5  # kN/m3

print("STEP 2: Bearing Capacity Analysis")
print("=" * 50)
print(f"Footing: {B_footing}m x {B_footing}m square at {D_footing}m depth")
print(f"Soil: c={c_design} kPa, phi={phi_design} deg, gamma={gamma_design} kN/m3")
print()

for shape in ["strip", "square", "circular"]:
    q_ult, q_allow = bearing_capacity(
        c_design, phi_design, gamma_design,
        B_footing, D_footing, shape
    )
    print(f"  {shape:10s}: q_ult = {q_ult:8.1f} kPa, "
          f"q_allow = {q_allow:8.1f} kPa (FoS=3)")

# Use square footing result for design check
q_ult_sq, q_allow_sq = bearing_capacity(
    c_design, phi_design, gamma_design,
    B_footing, D_footing, "square"
)

print()
applied_load = 150  # kPa
print(f"Applied pressure: {applied_load} kPa")
print(f"Allowable:        {q_allow_sq} kPa")
if applied_load <= q_allow_sq:
    print(f"Result: SAFE (FoS = {q_ult_sq/applied_load:.1f})")
else:
    print(f"Result: UNSAFE -- increase footing size or depth")

In [ ]:
# ================================================================
# STEP 3: Summary Report
# ================================================================
from statistics import mean

N60_values = [layer["N60"] for layer in layers]
Vs_values  = [layer["Vs"] for layer in layers]

print()
print("=" * 60)
print(f"  BOREHOLE SUMMARY REPORT -- {borehole['id']}")
print("=" * 60)
print(f"  Project:     {borehole['project']}")
print(f"  Location:    {borehole['location']}")
print(f"  Total depth: {layers[-1]['depth']} m")
print(f"  Water table: {borehole['water_table_m']} m")
print(f"  Layers:      {len(layers)}")
print()
print("  SPT Summary:")
print(f"    N60 range: {min(N60_values)} - {max(N60_values)}")
print(f"    N60 mean:  {mean(N60_values):.1f}")
print()
print("  Vs Summary:")
print(f"    Vs range:  {min(Vs_values)} - {max(Vs_values)} m/s")
print(f"    Vs mean:   {mean(Vs_values):.1f} m/s")
print()
print("  Soil Profile:")
for layer in layers:
    bar = '#' * (layer["N60"] // 3)
    print(f"    {layer['depth']:5.1f}m  N60={layer['N60']:3d}  "
          f"{layer['uscs']:>3s}  {layer['soil']:15s}  {bar}")
print()
print(f"  Footing Design ({B_footing}m x {B_footing}m square at {D_footing}m):")
print(f"    q_ult   = {q_ult_sq:.1f} kPa")
print(f"    q_allow = {q_allow_sq:.1f} kPa (FoS=3)")
print("=" * 60)
print()
print("All calculations used functions from our geotech_utils module.")
print("Write once, use everywhere -- that is the power of modules.")

---

## Practice Exercises

### Exercise 1: Add a function to the module
Write `elastic_settlement(q, B, E, mu)` that computes settlement using `S = q * B * (1 - mu^2) / E`. Test it with q=150 kPa, B=2m, E=25000 kPa, mu=0.3.

### Exercise 2: Use `statistics` on Vs values
Compute the mean, standard deviation, and coefficient of variation (CoV) of the Vs values from our borehole. If CoV > 30%, flag spatial variability.

### Exercise 3: Save results to JSON
Export the borehole analysis results (all layers with N60, Vs, USCS) to a JSON file using the `json` module. Load it back and print a summary.

In [ ]:
# EXERCISE 1
# def elastic_settlement(q_kPa, B_m, E_kPa, mu=0.3):
#     """Elastic settlement of a rigid footing on elastic half-space."""
#     S = q_kPa * B_m * (1 - mu**2) / E_kPa
#     return round(S * 1000, 2)  # convert m to mm
#
# Test:
# s = elastic_settlement(150, 2.0, 25000, 0.3)
# print(f"Settlement = {s} mm")

In [ ]:
# EXERCISE 2
# from statistics import mean, stdev
#
# Vs_list = [layer["Vs"] for layer in layers]
# avg = mean(Vs_list)
# sd = stdev(Vs_list)
# cov = sd / avg * 100
# print(f"Mean Vs: {avg:.1f} m/s")
# print(f"Stdev:   {sd:.1f} m/s")
# print(f"CoV:     {cov:.1f}%")
# if cov > 30:
#     print("High spatial variability!")

In [ ]:
# EXERCISE 3
# import json
#
# results = {
#     "borehole": borehole["id"],
#     "layers": [
#         {"depth": l["depth"], "N60": l["N60"],
#          "Vs": l["Vs"], "uscs": l["uscs"]}
#         for l in layers
#     ]
# }
# with open("project_data/bh04_results.json", "w") as f:
#     json.dump(results, f, indent=2)
#
# # Load and verify
# with open("project_data/bh04_results.json", "r") as f:
#     check = json.load(f)
# print(f"Loaded {len(check['layers'])} layers for {check['borehole']}")

---

## Module & Package Quick Reference

### Import Styles
```python
import math                    # full module  -->  math.sin()
from math import sin, cos     # specific     -->  sin()
import numpy as np            # alias        -->  np.array()
from math import *            # everything (avoid this!)
```

### Essential Standard Library for Engineers

| Module | Purpose | Key Functions |
|--------|---------|---------------|
| `math` | Math operations | `sin`, `cos`, `tan`, `sqrt`, `exp`, `log`, `pi` |
| `statistics` | Basic stats | `mean`, `median`, `stdev`, `variance` |
| `csv` | Read/write CSV | `reader`, `writer`, `DictReader`, `DictWriter` |
| `json` | Read/write JSON | `load`, `dump`, `loads`, `dumps` |
| `os` | OS interaction | `makedirs`, `listdir`, `path.exists` |
| `pathlib` | Modern paths | `Path`, `/` operator, `.glob()` |
| `datetime` | Dates and times | `datetime`, `timedelta`, `strftime` |

### Creating Your Own Module
```python
# Save as geotech_utils.py:
def my_function(): ...

if __name__ == "__main__":
    # Test code runs only when executed directly
    ...
```

---

## Key Takeaways

1. **Modules are `.py` files you import.** Python's Standard Library gives you `math`, `statistics`, `csv`, `json`, `os`, and `datetime` with no install needed.

2. **Three import styles** serve different needs: `import math` for many functions, `from math import sqrt` for just one, and `import numpy as np` for convenience.

3. **Create your own modules** by saving functions in a `.py` file. Write `spt_correction()` once, import it in every project. One source of truth.

4. **`__name__ == "__main__"`** lets your module work both as an importable library and as a standalone test script.

5. **Third-party packages** (`numpy`, `pandas`, `scipy`) extend Python for serious engineering. Install with `pip`, import with `as` aliases. We start using these daily from Phase 2.

---

## How This Connects to ML

| Today | ML Equivalent |
|-------|---------------|
| `import math` | `import torch`, `import tensorflow` |
| `from math import sqrt` | `from sklearn.ensemble import RandomForestClassifier` |
| `import numpy as np` | The universal convention in every ML tutorial |
| Custom `geotech_utils.py` | `model_utils.py`, `data_pipeline.py` in ML projects |
| `csv` / `json` for data | `pandas.read_csv()` -- built on the same ideas |
| `__name__ == "__main__"` | Training scripts that are also importable modules |
| `pip install` | Installing ML frameworks, experiment trackers, data loaders |

**Every ML project is built from modules.** A typical structure:
```
soil_classifier/
    data_loader.py      <-- loads and cleans borehole CSVs
    features.py         <-- computes N60, Vs, grain-size features
    model.py            <-- defines and trains the classifier
    evaluate.py         <-- computes accuracy, confusion matrix
    utils.py            <-- shared helpers (plotting, logging)
```
Today you learned the pattern that makes this possible.

---

*Day 012 of 100 -- Write once, import everywhere.*